# ETF Portfolio Backtester - Demo Notebook

**DADS 4002 Course Project**

This notebook demonstrates how to use the ETF Portfolio Backtester system interactively.

---

## 📦 Step 1: Import Libraries and Modules

In [ ]:
import sys
import os
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

# Import project modules
from modules.db_connector import DatabaseConnector
from modules.backtest_engine import BacktestEngine
from modules.analytics import Analytics
from modules.crud_operations import CRUDOperations

# Set pandas display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

print("✓ All modules imported successfully!")

## 🔌 Step 2: Connect to Database

In [ ]:
# Initialize database connection
db = DatabaseConnector()

# Test connection
if db.test_connection():
    print("✓ Database connected successfully!\n")
    
    # Check data
    etf_count = db.get_table_count('ETF_Master')
    price_count = db.get_table_count('Price_Data')
    
    print(f"✓ ETF_Master: {etf_count} ETFs")
    print(f"✓ Price_Data: {price_count:,} records")
else:
    print("✗ Cannot connect to database")
    print("Please check MySQL connection settings in modules/db_connector.py")

## 📊 Step 3: View ETF Master Data

In [ ]:
# Query all ETFs
query = """
SELECT
    ETF_ID,
    Ticker_Symbol,
    ETF_Name,
    Asset_Type,
    Expense_Ratio
FROM ETF_Master
ORDER BY Asset_Type, Ticker_Symbol
"""

results = db.execute_query_dict(query)
df_etfs = pd.DataFrame(results)

print(f"Total ETFs: {len(df_etfs)}\n")

# Display first 10
print("Sample ETFs:")
print(df_etfs.head(10))

# Count by Asset Type
print("\nETF Count by Asset Type:")
print(df_etfs.groupby('Asset_Type').size())

## 📈 Step 4: View Recent Price Data (SPY Example)

In [ ]:
# Query SPY recent prices
query = """
SELECT
    pd.Price_Date,
    pd.Open_Price,
    pd.High_Price,
    pd.Low_Price,
    pd.Close_Price,
    pd.Volume,
    em.Ticker_Symbol
FROM Price_Data pd
JOIN ETF_Master em ON pd.ETF_ID = em.ETF_ID
WHERE em.Ticker_Symbol = 'SPY'
ORDER BY pd.Price_Date DESC
LIMIT 10
"""

results = db.execute_query_dict(query)
df_spy = pd.DataFrame(results)

print("SPY - Latest 10 Weeks:")
print(df_spy)

# Plot
df_spy_sorted = df_spy.sort_values('Price_Date')
plt.figure(figsize=(12, 6))
plt.plot(df_spy_sorted['Price_Date'], df_spy_sorted['Close_Price'], marker='o', linewidth=2)
plt.title('SPY - Close Price (Last 10 Weeks)', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Price ($)')
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 🎯 Step 5: Run Backtest

In [ ]:
# Initialize backtest engine
backtest = BacktestEngine(db)

# Set parameters
selection_date = datetime.now()
lookback_days = 90
top_n = 5
holding_days = 30

print("Running Backtest...")
print("=" * 60)
print(f"Selection Date: {selection_date.date()}")
print(f"Lookback Period: {lookback_days} days")
print(f"Top N ETFs: {top_n}")
print(f"Holding Period: {holding_days} days")
print("=" * 60)

# Run backtest
results = backtest.run_backtest(
    selection_date=selection_date,
    lookback_days=lookback_days,
    top_n=top_n,
    holding_days=holding_days
)

# Display results
if results:
    print("\n✓ Backtest completed successfully!")
    print(f"Backtest Run ID: {results['backtest_run_id']}")
    
    # Portfolio details
    print(f"\nTop {top_n} ETFs Selected:")
    df_results = pd.DataFrame(results['portfolio'])
    print(df_results[['rank', 'ticker', 'etf_name', 'asset_type', 'momentum_score', 'holding_return']])
    
    # Portfolio return
    print(f"\n{'='*60}")
    print(f"Portfolio Return: {results['portfolio_return']:.2f}%")
    print(f"{'='*60}")
else:
    print("✗ Backtest failed")

## 📊 Step 6: Analytics - Volatility Analysis

In [ ]:
# Initialize analytics
analytics = Analytics(db)

print("Insight #1: Volatility Analysis by Asset Type")
print("=" * 60)

# Get volatility data
volatility_df = analytics.get_volatility_analysis()

if not volatility_df.empty:
    print(volatility_df)
    
    # Plot
    plt.figure(figsize=(10, 6))
    plt.bar(volatility_df['Asset_Type'], volatility_df['Annualized_Volatility'], 
            color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'])
    plt.title('Annualized Volatility by Asset Type', fontsize=14, fontweight='bold')
    plt.xlabel('Asset Type')
    plt.ylabel('Annualized Volatility')
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print("\n💡 Insight: Higher volatility = Higher risk")
else:
    print("No data available")

## 📊 Step 7: Analytics - Lookback Period Optimization

In [ ]:
print("Insight #2: Lookback Period Optimization")
print("=" * 60)

# Get lookback comparison
lookback_df = analytics.get_lookback_comparison()

if not lookback_df.empty:
    print(lookback_df)
    
    # Plot
    plt.figure(figsize=(10, 6))
    plt.plot(lookback_df['Lookback_Days'], lookback_df['CAGR'], 
             marker='o', linewidth=2, markersize=8, color='#2ca02c')
    plt.title('CAGR by Lookback Period', fontsize=14, fontweight='bold')
    plt.xlabel('Lookback Period (Days)')
    plt.ylabel('CAGR (%)')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Find best lookback
    best_row = lookback_df.loc[lookback_df['CAGR'].idxmax()]
    print(f"\n💡 Insight: Best Lookback Period = {best_row['Lookback_Days']} days (CAGR: {best_row['CAGR']:.2f}%)")
else:
    print("No backtest data available. Run some backtests first!")

## 📊 Step 8: Analytics - Drawdown Analysis

In [ ]:
print("Insight #3: Drawdown Analysis")
print("=" * 60)

# Get drawdown analysis
drawdown_df = analytics.get_drawdown_analysis()

if not drawdown_df.empty:
    print(drawdown_df)
    
    # Plot
    plt.figure(figsize=(10, 6))
    plt.bar(drawdown_df['Asset_Type'], drawdown_df['Holdings_During_Drawdown'],
            color=['#d62728', '#ff7f0e', '#1f77b4', '#2ca02c'])
    plt.title('Asset Holdings During Drawdown Periods', fontsize=14, fontweight='bold')
    plt.xlabel('Asset Type')
    plt.ylabel('Number of Holdings')
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print("\n💡 Insight: Which asset types were held during worst performance periods")
else:
    print("No backtest data available")

## 🔍 Step 9: CRUD Operations - View Latest Backtest

In [ ]:
# Initialize CRUD operations
crud = CRUDOperations(db)

print("Latest Backtest Results:")
print("=" * 60)

# Read latest backtest
results = crud.read_latest_backtest()

if results:
    df = pd.DataFrame(results)
    print(df[['rank', 'ticker', 'etf_name', 'momentum_score', 'holding_return']])
    
    print(f"\nTotal ETFs in portfolio: {len(df)}")
    print(f"Average Return: {df['holding_return'].mean():.2f}%")
else:
    print("No backtest results found")

## 📈 Step 10: Advanced Analysis - Cumulative Returns

In [ ]:
# Get 1 year of data for top 5 ETFs
query = """
SELECT
    em.Ticker_Symbol,
    em.Asset_Type,
    pd.Price_Date,
    pd.Close_Price
FROM Price_Data pd
JOIN ETF_Master em ON pd.ETF_ID = em.ETF_ID
WHERE pd.Price_Date >= DATE_SUB(CURDATE(), INTERVAL 1 YEAR)
  AND em.Ticker_Symbol IN ('SPY', 'QQQ', 'AGG', 'GLD', 'TLT')
ORDER BY em.Ticker_Symbol, pd.Price_Date
"""

results = db.execute_query_dict(query)
df = pd.DataFrame(results)

if not df.empty:
    # Pivot table
    pivot = df.pivot_table(
        index='Price_Date',
        columns='Ticker_Symbol',
        values='Close_Price'
    )
    
    # Calculate returns
    returns = pivot.pct_change()
    
    # Calculate cumulative returns
    cumulative_returns = (1 + returns).cumprod()
    
    # Plot
    plt.figure(figsize=(14, 7))
    
    for ticker in cumulative_returns.columns:
        plt.plot(cumulative_returns.index, cumulative_returns[ticker], 
                label=ticker, linewidth=2)
    
    plt.title('Cumulative Returns - Last 1 Year', fontsize=14, fontweight='bold')
    plt.xlabel('Date')
    plt.ylabel('Cumulative Return')
    plt.legend(loc='best')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Summary statistics
    print("\nSummary Statistics (Annualized):")
    annual_returns = returns.mean() * 52
    annual_vol = returns.std() * (52 ** 0.5)
    
    summary = pd.DataFrame({
        'Annual Return': annual_returns,
        'Annual Volatility': annual_vol,
        'Sharpe Ratio': annual_returns / annual_vol
    })
    
    print(summary)
else:
    print("No data available")

## 🔒 Step 11: Close Database Connection

In [ ]:
# Close database connection
db.close_pool()
print("✓ Database connection closed successfully")
print("\nThank you for using ETF Portfolio Backtester!")

---

## 📚 Notes

- **All data is REAL from Yahoo Finance** (not synthetic)
- **50 real ETFs** across 4 asset types
- **10 years** of weekly price data
- **~25,933 price records** in total

---

**DADS 4002 Course Project** | Made with ❤️